In [ ]:
import os
import json
import subprocess
import numpy as np
import pandas as pd

In [ ]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
TYPES = ["attribute", "relation", "object"]
CONTROL_RATIO = 1.5
MIN_CONTROLS = 300
SHARES = {"true_yes": 0.6, "true_no": 0.4}
PLACEHOLDER_SEC_PER_QUESTION = 2.0

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/vlm-labeled-examples/labeled_examples.json") as f:
    labels = pd.DataFrame(json.load(f))
assert "cache" in labels.columns, "attach vlm-labeled-examples version 3"
rec = labels[labels["cache"] == "question_aware"][["question_id", "source", "split", "model_answer"]].copy()
rec["type"] = rec.question_id.map(lambda q: question_by_id[q]["hallucination_type"])
rec["truth"] = rec.question_id.map(lambda q: question_by_id[q]["ground_truth_answer"].strip().lower())
rec["group"] = np.select(
    [
        (rec.truth == "yes") & (rec.model_answer == "yes"),
        (rec.truth == "no") & (rec.model_answer == "no"),
        (rec.truth == "no") & (rec.model_answer == "yes"),
        (rec.truth == "yes") & (rec.model_answer == "no"),
    ],
    ["true_yes", "true_no", "false_yes", "false_no"],
    default="other",
)
assert (rec.group != "other").all()
assert rec.question_id.is_unique
print(len(rec), "labeled yes/no questions")
print(pd.crosstab([rec.type, rec.source, rec.split], rec.group))

In [ ]:
parts = []
for t in TYPES:
    d = rec[rec.type == t]
    fy = d[d.group == "false_yes"]
    parts.append(fy.assign(stratum="false_yes"))
    budget = max(int(CONTROL_RATIO * len(fy)), MIN_CONTROLS)
    for g, share in SHARES.items():
        pool = d[d.group == g]
        if len(pool) == 0:
            continue
        quota = min(int(budget * share), len(pool))
        parts.append(pool.groupby(["source", "split"], group_keys=False).sample(frac=quota / len(pool), random_state=0).assign(stratum=g))

selection = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
assert selection.question_id.is_unique
assert (selection.stratum == "false_yes").sum() == int((rec.group == "false_yes").sum()) - int(((rec.group == "false_yes") & ~rec.type.isin(TYPES)).sum()), "false_yes rows were dropped"

print(pd.crosstab([selection.type, selection.source, selection.split], selection.stratum, margins=True))
sel_counts = selection.groupby("type").size().to_dict()
print("selected per type:", sel_counts, "of", rec[rec.type.isin(TYPES)].groupby("type").size().to_dict())
print(f"{len(selection)} questions at a placeholder {PLACEHOLDER_SEC_PER_QUESTION} s/question: about "
      f"{len(selection) * PLACEHOLDER_SEC_PER_QUESTION / 3600:.1f} h of collection (replace with the measured speed)")

In [ ]:
SEL_DIR = "/kaggle/working/atp_llava_selection"
os.makedirs(SEL_DIR, exist_ok=True)

with open(os.path.join(SEL_DIR, "selection.json"), "w") as f:
    json.dump({
        "selected": {t: selection[selection.type == t].question_id.tolist() for t in TYPES},
        "stratum": dict(zip(selection.question_id, selection.stratum)),
        "group": dict(zip(selection.question_id, selection.group)),
        "source": dict(zip(selection.question_id, selection.source)),
        "split": dict(zip(selection.question_id, selection.split)),
    }, f)

with open(os.path.join(SEL_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "atp-llava-selection",
        "id": f"{USERNAME}/atp-llava-selection",
        "subtitle": "Question subsample for LLaVA attribution patching",
        "description": "Questions selected for LLaVA-1.5-7B attribution patching from the question-aware labels: every false_yes row "
                       "(model said yes, truth no) plus stratified true_yes (60%) and true_no (40%) controls by source and split, "
                       "about 1.5 controls per false_yes row for each of attribute, relation and object. "
                       "stratum, group, source and split map each question_id to its selection group, label group, dataset and split.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


ok, out = _run(["kaggle", "datasets", "create", "-p", SEL_DIR])
if not ok:
    ok, out2 = _run(["kaggle", "datasets", "version", "-p", SEL_DIR, "-m", "selection"])
    out += "\n" + out2
print(out)
assert ok, "dataset upload failed"